In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
from pathlib import Path

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_chroma import Chroma

from langchain_core.prompts import PromptTemplate

from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser

api_key = os.getenv("GOOGLE_API_KEY")
CHROMA_DIR = Path("../chroma_db")

print(f"API key loaded: {api_key[:8]}...")
print(f"ChromaDB path exists: {CHROMA_DIR.exists()}")


API key loaded: AQ.Ab8RN...
ChromaDB path exists: True


In [ ]:

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=api_key
)

vectorstore = Chroma(
    collection_name="ml_papers",       
    persist_directory=str(CHROMA_DIR), 
    embedding_function=embeddings     
)


retriever = vectorstore.as_retriever(search_kwargs={"k":5})


test_docs = retriever.invoke("What is a residual connection?")
print(f"Retrieved {len(test_docs)} chunks\n")


for doc in test_docs:
    print(f"Source: {doc.metadata['source']} | Page: {doc.metadata['page']}")



Retrieved 5 chunks

Source: resnet | Page: 3
Source: resnet | Page: 2
Source: resnet | Page: 2
Source: resnet | Page: 3
Source: resnet | Page: 2


In [ ]:


prompt_template = """
You are a helpful research assistant that explains machine learning papers 
to third-year undergraduate students in plain English.

Use ONLY the information provided in the context below to answer the question.
Do NOT use any outside knowledge.

For every piece of information you use, cite the source paper and page number 
in this format: [Paper: source_name, Page: X]

If the context contains mathematical notation or equations that are hard to read,
describe what the equation means in plain English instead of copying it directly.

If the context does not contain enough information to answer the question,
say: "I don't have enough information in the indexed papers to answer this question."

Context:
{context}

Question:
{question}

Answer (in plain English, with citations):
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"] 
)

print("Prompt template ready.")
print(f"Input variables: {prompt.input_variables}")


Prompt template ready.
Input variables: ['context', 'question']


In [ ]:

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.2
)

def format_docs(docs):
    formatted=[]
    for doc in docs:
        
        chunk_text = f"[Paper: {doc.metadata['source']}, Page: {doc.metadata['page']}]\n{doc.page_content}"
        formatted.append(chunk_text)
    
    return "\n\n---\n\n".join(formatted)



rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


RAG chain built successfully.


In [ ]:

question = "What problem does the Transformer architecture solve that previous models couldn't?"

print(f"Question: {question}")
print("="*60)

answer = rag_chain.invoke(question)

print(answer)

Question: What problem does the Transformer architecture solve that previous models couldn't?
The Transformer architecture solves two main problems that previous models faced:

1.  **Lack of Parallelization and Sequential Computation:** Previous models, particularly those based on recurrent networks, had a "sequential nature" that prevented parallel processing within training examples. This became especially problematic with longer sequences, as memory limitations restricted the ability to process multiple examples at once [Paper: attention_is_all_you_need, Page: 2]. The Transformer addresses this by "eschewing recurrence" and instead relying entirely on an attention mechanism, which "allows for significantly more parallelization" [Paper: attention_is_all_you_need, Page: 2]. This makes the Transformer "more parallelizable and requiring significantly less time to train" [Paper: attention_is_all_you_need, Page: 1].

2.  **Difficulty Learning Long-Range Dependencies:** Models like ConvS2S

In [ ]:
import gradio as gr

def answer_question(question, chat_history):

    
    if not question.strip():
        return "", chat_history
    
    answer = rag_chain.invoke(question)

   
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    return "", chat_history

with gr.Blocks(title="ML Research Paper Chatbot") as demo:

    gr.Markdown( "# 📚 ML Research Paper Chatbot")
    gr.Markdown(
        "Ask questions about 12 landmark ML papers including Attention Is All You Need, "
        "BERT, ResNet, GANs, RAG, and more. Every answer is cited with the source paper and page."
    )

    with gr.Accordion("📄 Indexed Papers — click to see the full list", open=False):
        gr.Markdown("""
        | # | Paper | Year |
        |---|-------|------|
        | 1 | Attention Is All You Need (Transformer) | 2017 |
        | 2 | BERT | 2018 |
        | 3 | ResNet | 2015 |
        | 4 | GAN — Generative Adversarial Networks | 2014 |
        | 5 | Vision Transformer (ViT) | 2020 |
        | 6 | RAG — Retrieval-Augmented Generation | 2020 |
        | 7 | Word2Vec | 2013 |
        | 8 | Adam Optimizer | 2014 |
        | 9 | Batch Normalization | 2015 |
        | 10 | Dropout | 2014 |
        | 11 | Deep RL — Atari | 2015 |
        | 12 | Scaling Laws | 2020 |
        """)

    chatbot = gr.Chatbot(
        label="Chat",
        height=500,         
    )


    with gr.Row():

    
        question_box = gr.Textbox(
            placeholder="e.g. What is self-attention and why does it matter?",
            label="Your Question",
            scale=8
        )

        submit_btn = gr.Button("Ask", variant="primary", scale=2)

    gr.Examples(
        examples=[
            "What problem does the Transformer architecture solve?",
            "How does BERT differ from GPT?",
            "What is a residual connection and why does it help?",
            "How do GANs work?",
            "What is the RAG architecture?",
            "Why does the Adam optimizer work better than SGD?",
        ],
        inputs=question_box  
    )

    submit_btn.click(
        fn=answer_question,                    
        inputs= [question_box, chatbot],         
        outputs=[question_box, chatbot]         
    )

    
    question_box.submit(
        fn=answer_question,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    )

print("Gradio interface built. Ready to launch")





Gradio interface built. Ready to launch


In [17]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
